# Movie Recommendation System — Full Data Science Workflow

**Dataset:** MovieLens Small (100K ratings, 9 742 movies, 610 users)  
**Models:** SVD Collaborative Filtering · TF-IDF Content-Based · Hybrid  

## Table of Contents
1. [Setup & Imports](#1-setup)
2. [Exploratory Data Analysis](#2-eda)
3. [Feature Engineering](#3-features)
4. [Model Training](#4-training)
5. [Evaluation](#5-evaluation)
6. [Monitoring & Drift Detection](#6-monitoring)
7. [Recommendation Examples](#7-examples)

## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
import os, sys
# Make sure the project root is on the path
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import (
    load_movies, load_ratings, load_tags,
    train_test_split_temporal, filter_min_interactions
)
from src.models import CollaborativeFilter, ContentBasedFilter, HybridRecommender
from src.evaluation import (
    evaluate_rating_prediction, evaluate_ranking, catalogue_coverage
)
from src.monitoring import ModelMonitor

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

## 2. Exploratory Data Analysis <a id='2-eda'></a>

In [ ]:
movies  = load_movies()
ratings = load_ratings()
tags    = load_tags()

print(f'Movies : {len(movies):,} rows   Columns: {list(movies.columns)}')
print(f'Ratings: {len(ratings):,} rows   Columns: {list(ratings.columns)}')
print(f'Tags   : {len(tags):,} rows   Columns: {list(tags.columns)}')

In [ ]:
movies.head(10)

In [ ]:
ratings.head(10)

### 2.1 Basic statistics

In [ ]:
print('=== Ratings ===')
print(ratings['rating'].describe())
print(f'\nUnique users : {ratings["userId"].nunique()}')
print(f'Unique movies: {ratings["movieId"].nunique()}')
print(f'Sparsity     : {1 - len(ratings) / (ratings["userId"].nunique() * ratings["movieId"].nunique()):.2%}')

### 2.2 Rating distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of ratings
ratings['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Ratings over time
ratings.set_index('timestamp').resample('M')['rating'].count().plot(
    ax=axes[1], color='coral'
)
axes[1].set_title('Ratings per Month')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Number of ratings')

plt.tight_layout()
plt.show()

### 2.3 User & movie activity (long-tail effect)

In [ ]:
user_counts  = ratings.groupby('userId')['movieId'].count().sort_values(ascending=False)
movie_counts = ratings.groupby('movieId')['userId'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# User activity
axes[0].plot(range(len(user_counts)), user_counts.values, color='steelblue')
axes[0].set_title('Ratings per User (long-tail)')
axes[0].set_xlabel('User rank')
axes[0].set_ylabel('# ratings')
axes[0].set_yscale('log')

# Movie popularity
axes[1].plot(range(len(movie_counts)), movie_counts.values, color='coral')
axes[1].set_title('Ratings per Movie (long-tail)')
axes[1].set_xlabel('Movie rank')
axes[1].set_ylabel('# ratings')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f'Median ratings per user : {user_counts.median():.0f}')
print(f'Top-10% users account for {user_counts.nlargest(int(len(user_counts)*0.1)).sum() / user_counts.sum():.1%} of all ratings')
print(f'Top-10% movies account for {movie_counts.nlargest(int(len(movie_counts)*0.1)).sum() / movie_counts.sum():.1%} of all ratings')

### 2.4 Genre distribution

In [ ]:
# Explode genres (one row per genre)
genre_series = movies['genres'].str.split('|').explode()
genre_counts = genre_series[genre_series != '(no genres listed)'].value_counts()

plt.figure(figsize=(10, 5))
genre_counts.plot(kind='barh', color='mediumpurple', edgecolor='white')
plt.title('Number of Movies per Genre')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

### 2.5 Average rating per genre

In [ ]:
# Join ratings with movies, then explode genres
merged = ratings.merge(movies[['movieId', 'genres']], on='movieId')
merged_exploded = merged.copy()
merged_exploded['genre'] = merged_exploded['genres'].str.split('|')
merged_exploded = merged_exploded.explode('genre')
merged_exploded = merged_exploded[merged_exploded['genre'] != '(no genres listed)']

genre_avg = (
    merged_exploded.groupby('genre')['rating']
    .agg(['mean', 'count'])
    .query('count >= 500')   # only genres with enough ratings
    .sort_values('mean')
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(genre_avg.index, genre_avg['mean'], color='teal', edgecolor='white')
ax.axvline(ratings['rating'].mean(), color='red', linestyle='--', label=f'Overall mean ({ratings["rating"].mean():.2f})')
ax.set_title('Average Rating by Genre (min 500 ratings)')
ax.set_xlabel('Average Rating')
ax.legend()
plt.tight_layout()
plt.show()

### 2.6 Release year trends

In [ ]:
movies_with_year = movies.dropna(subset=['year'])
year_counts = movies_with_year['year'].value_counts().sort_index()

merged_year = ratings.merge(movies[['movieId', 'year']].dropna(), on='movieId')
year_avg_rating = merged_year.groupby('year')['rating'].agg(['mean', 'count'])
year_avg_rating = year_avg_rating[year_avg_rating['count'] >= 100]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

year_counts[year_counts.index >= 1960].plot(ax=axes[0], color='steelblue')
axes[0].set_title('Movies Released per Year (since 1960)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

year_avg_rating['mean'].plot(ax=axes[1], color='coral')
axes[1].set_title('Average Rating by Release Year (min 100 ratings)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Average Rating')

plt.tight_layout()
plt.show()

## 3. Feature Engineering <a id='3-features'></a>

### 3.1 Collaborative Filtering features — User-Item Matrix

The raw material for CF is the **user–item rating matrix**.  
We inspect its sparsity and distribution before feeding it to SVD.

In [ ]:
from src.data_loader import build_user_item_matrix, filter_min_interactions

ratings_filtered = filter_min_interactions(ratings, min_user_ratings=5, min_movie_ratings=5)
uim = build_user_item_matrix(ratings_filtered)

n_users, n_movies = uim.shape
sparsity = 1 - uim.notna().sum().sum() / (n_users * n_movies)

print(f'User-item matrix shape : {n_users} × {n_movies}')
print(f'Sparsity               : {sparsity:.2%}')
print(f'Density                : {1-sparsity:.2%}')

In [ ]:
# Visualise a small slice of the matrix
sample = uim.iloc[:30, :50].fillna(0)

plt.figure(figsize=(14, 5))
sns.heatmap(sample, cmap='YlOrRd', linewidths=0, cbar_kws={'label': 'Rating'})
plt.title('User-Item Rating Matrix (first 30 users × 50 movies)')
plt.xlabel('Movie index')
plt.ylabel('User index')
plt.tight_layout()
plt.show()

### 3.2 User-level features

In [ ]:
user_features = ratings_filtered.groupby('userId').agg(
    n_ratings    = ('rating', 'count'),
    mean_rating  = ('rating', 'mean'),
    std_rating   = ('rating', 'std'),
    min_rating   = ('rating', 'min'),
    max_rating   = ('rating', 'max'),
    first_rating = ('timestamp', 'min'),
    last_rating  = ('timestamp', 'max'),
).reset_index()
user_features['active_days'] = (
    user_features['last_rating'] - user_features['first_rating']
).dt.days
user_features.head()

### 3.3 Content-based features — Genre one-hot encoding

In [ ]:
# One-hot encode genres
genre_dummies = movies['genres'].str.get_dummies(sep='|')
movies_encoded = pd.concat([movies[['movieId', 'title', 'year']], genre_dummies], axis=1)

print(f'Genre columns: {list(genre_dummies.columns)}')
movies_encoded.head(5)

### 3.4 TF-IDF representation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

movies['genres_text']  = movies['genres'].str.replace('|', ' ', regex=False)
movies['title_clean']  = movies['title'].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()
movies['content_text'] = movies['genres_text'] + ' ' + movies['title_clean']

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(movies['content_text'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Sample top tokens: {list(tfidf.vocabulary_.keys())[:20]}')

## 4. Model Training <a id='4-training'></a>

In [ ]:
train_ratings, test_ratings = train_test_split_temporal(ratings_filtered, test_ratio=0.2)
print(f'Train: {len(train_ratings):,} ratings  |  Test: {len(test_ratings):,} ratings')

### 4.1 Collaborative Filter — SVD Matrix Factorisation

We factorise the mean-centred user–item matrix with **TruncatedSVD** (k=50 latent factors).  
The dot product of user and item factor vectors gives the predicted preference deviation,  
to which we add back the user's mean rating to recover the predicted rating scale.

In [ ]:
import time
t0 = time.time()
cf = CollaborativeFilter(n_factors=50, random_state=42)
cf.fit(train_ratings)
print(f'CF trained in {time.time()-t0:.1f}s')

In [ ]:
# How much variance do the latent factors explain?
explained_var = cf._svd.explained_variance_ratio_
cumulative = explained_var.cumsum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, len(explained_var)+1), explained_var, alpha=0.6, color='steelblue', label='Individual')
ax2 = ax.twinx()
ax2.plot(range(1, len(cumulative)+1), cumulative, color='red', marker='o', markersize=3, label='Cumulative')
ax2.axhline(0.9, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Latent Factor Index')
ax.set_ylabel('Explained Variance Ratio')
ax2.set_ylabel('Cumulative Explained Variance')
ax.set_title('SVD — Explained Variance per Latent Factor')
ax.legend(loc='upper right')
ax2.legend(loc='center right')
plt.tight_layout()
plt.show()
print(f'Top 10 factors explain {cumulative[9]:.1%} of variance')

### 4.2 Content-Based Filter — TF-IDF + Cosine Similarity

Each movie is represented as a TF-IDF vector over genre tokens + title words.  
Recommendations are the cosine-nearest neighbours in this vector space.

In [ ]:
t0 = time.time()
cbf = ContentBasedFilter(max_features=5000)
cbf.fit(movies)
print(f'CBF trained in {time.time()-t0:.1f}s')

### 4.3 Hybrid Recommender

Combines CF (70%) and CBF (30%) after min-max normalising both score sets.  
This reduces the cold-start penalty and tempers CF popularity bias.

In [ ]:
t0 = time.time()
hybrid = HybridRecommender(cf_weight=0.7, cbf_weight=0.3)
hybrid.fit(train_ratings, movies)
print(f'Hybrid trained in {time.time()-t0:.1f}s')

## 5. Evaluation <a id='5-evaluation'></a>

### 5.1 Rating Prediction Quality (CF)

RMSE and MAE measure how close the model's predicted ratings are to the actual ratings.  
Lower is better. A baseline of predicting the global mean gives RMSE ≈ 1.06 on this dataset.

In [ ]:
rating_metrics = evaluate_rating_prediction(cf, test_ratings, sample_size=2000)
print('Collaborative Filter — Rating Prediction')
for k, v in rating_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

### 5.2 Baseline comparison — predict global mean

In [ ]:
from src.evaluation import rmse, mae

global_mean = train_ratings['rating'].mean()
baseline_preds = [global_mean] * len(test_ratings)

baseline_rmse = rmse(test_ratings['rating'].tolist(), baseline_preds)
baseline_mae  = mae(test_ratings['rating'].tolist(), baseline_preds)

print(f'Global-mean baseline — RMSE: {baseline_rmse:.4f}, MAE: {baseline_mae:.4f}')
print(f'CF improvement       — RMSE: {(baseline_rmse - rating_metrics["RMSE"]):.4f} better')

### 5.3 Ranking Quality (CF)

**Precision@K** — of the K items shown, how many are actually relevant?  
**Recall@K**    — of all relevant items, how many appear in the top K?  
**NDCG@K**      — did the model rank relevant items near the top?  
**HitRate@K**   — did the user find at least one relevant item in top K?  

"Relevant" = rated ≥ 3.5 in the test set.

In [ ]:
ranking_metrics = evaluate_ranking(cf, test_ratings, k=10, n_users=100)
print('Collaborative Filter — Ranking Quality @K=10')
for k, v in ranking_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

### 5.4 Metrics vs K — finding the best list length

In [ ]:
k_values = [5, 10, 15, 20]
results_by_k = []
for k in k_values:
    m = evaluate_ranking(cf, test_ratings, k=k, n_users=50)
    m['k'] = k
    results_by_k.append(m)

df_k = pd.DataFrame(results_by_k).set_index('k')
# Drop count column if present
df_k = df_k[[c for c in df_k.columns if c != 'n_users_evaluated']]

fig, ax = plt.subplots(figsize=(10, 4))
for col in df_k.columns:
    ax.plot(df_k.index, df_k[col], marker='o', label=col)
ax.set_title('Ranking Metrics vs K')
ax.set_xlabel('K')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.show()

### 5.5 Catalogue coverage

In [ ]:
sample_users = train_ratings['userId'].unique()[:200].tolist()
cf_cov  = catalogue_coverage(cf,  sample_users, total_items=len(movies))
cbf_cov = catalogue_coverage(cbf, [m for m in movies['movieId'].tolist()[:200]], total_items=len(movies))

print(f'CF  catalogue coverage (@10, 200 users)  : {cf_cov:.2%}')
print(f'CBF catalogue coverage (@10, 200 queries): {cbf_cov:.2%}')
print()
print('Low coverage = model is recommending only popular items (popularity bias).')
print('High coverage = more diverse recommendations but potentially lower precision.')

## 6. Monitoring & Drift Detection <a id='6-monitoring'></a>

In [ ]:
monitor = ModelMonitor('collaborative_filter_v1', alert_rmse_threshold=1.2)

# Simulate two evaluation rounds
from src.evaluation import rmse
import numpy as np

rng = np.random.default_rng(42)

for i, label in enumerate(['week_1', 'week_2', 'week_3']):
    # Simulate slight degradation over time
    noise = rng.normal(0, 0.05 * (i + 1), 500)
    y_true = rng.uniform(1, 5, 500)
    y_pred = np.clip(y_true + noise, 0.5, 5.0)
    monitor.log_performance(y_true.tolist(), y_pred.tolist(), label=label)

print(monitor.summary())

In [ ]:
perf_df = monitor.get_performance_df()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
perf_df.plot(x='label', y='RMSE', kind='bar', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('RMSE over Time')
axes[0].set_xlabel('')

perf_df.plot(x='label', y='MAE', kind='bar', ax=axes[1], color='coral', legend=False)
axes[1].set_title('MAE over Time')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Drift detection — compare rating distributions from two time windows
train_ratings_sorted = train_ratings.sort_values('timestamp')
n = len(train_ratings_sorted)

first_half  = train_ratings_sorted.iloc[:n//2]['rating']
second_half = train_ratings_sorted.iloc[n//2:]['rating']

drift_result = monitor.detect_data_drift(
    reference=first_half,
    current=second_half,
    label='rating_distribution',
)
print('Drift detection result:')
for k, v in drift_result.items():
    print(f'  {k}: {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
first_half.value_counts().sort_index().plot(
    kind='bar', ax=ax, alpha=0.6, color='steelblue', label='First half'
)
second_half.value_counts().sort_index().plot(
    kind='bar', ax=ax, alpha=0.6, color='coral', label='Second half'
)
ax.set_title('Rating Distribution: First Half vs Second Half (temporal)')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Recommendation Examples <a id='7-examples'></a>

### 7.1 Collaborative filtering — top-10 for a user

In [ ]:
USER_ID = 1

recs = cf.recommend(USER_ID, n=10)
rec_ids = [mid for mid, _ in recs]
rec_scores = [s for _, s in recs]

rec_df = movies[movies['movieId'].isin(rec_ids)].copy()
rec_df = rec_df.set_index('movieId').loc[rec_ids].reset_index()
rec_df['predicted_rating'] = [round(s, 2) for s in rec_scores]

print(f'Top-10 CF recommendations for user {USER_ID}:')
rec_df[['title', 'genres', 'predicted_rating']]

In [ ]:
# Show what this user has already rated highly
user_seen = ratings[ratings['userId'] == USER_ID].merge(movies[['movieId', 'title', 'genres']], on='movieId')
print(f'User {USER_ID} — top rated movies:')
user_seen.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(10)

### 7.2 Content-based — similar movies

In [ ]:
QUERY_TITLE = 'Toy Story'
query_row = movies[movies['title_clean'].str.contains(QUERY_TITLE, case=False, na=False)].iloc[0]
QUERY_MOVIE_ID = int(query_row['movieId'])

print(f'Query movie: {query_row["title"]} — genres: {query_row["genres"]}')
print()

cbf_recs = cbf.recommend(QUERY_MOVIE_ID, n=10)
cbf_ids = [mid for mid, _ in cbf_recs]
cbf_scores = [s for _, s in cbf_recs]

cbf_df = movies[movies['movieId'].isin(cbf_ids)].copy()
cbf_df = cbf_df.set_index('movieId').loc[cbf_ids].reset_index()
cbf_df['similarity'] = [round(s, 3) for s in cbf_scores]

print('Top-10 content-based similar movies:')
cbf_df[['title', 'genres', 'similarity']]

### 7.3 Hybrid recommendations

In [ ]:
hybrid_recs = hybrid.recommend_for_user(USER_ID, n=10)
h_ids = [mid for mid, _ in hybrid_recs]
h_scores = [s for _, s in hybrid_recs]

h_df = movies[movies['movieId'].isin(h_ids)].copy()
h_df = h_df.set_index('movieId').reindex(h_ids).reset_index()
h_df['hybrid_score'] = [round(s, 3) for s in h_scores]

print(f'Top-10 Hybrid recommendations for user {USER_ID}:')
h_df[['title', 'genres', 'hybrid_score']]

### 7.4 Model comparison — overlap between CF and Hybrid

In [ ]:
cf_set = set(rec_ids)
hybrid_set = set(h_ids)
overlap = cf_set & hybrid_set

print(f'CF recommendations    : {len(cf_set)} movies')
print(f'Hybrid recommendations: {len(hybrid_set)} movies')
print(f'Overlap               : {len(overlap)} movies ({len(overlap)/10:.0%})')
print()
print('Movies unique to Hybrid (CBF contribution):')
unique_hybrid = hybrid_set - cf_set
if unique_hybrid:
    movies[movies['movieId'].isin(unique_hybrid)][['title', 'genres']]

## Summary

| Model | RMSE | MAE | Precision@10 | Recall@10 | NDCG@10 |
|---|---|---|---|---|---|
| Global mean baseline | ~1.06 | ~0.84 | — | — | — |
| Collaborative Filter (SVD k=50) | see above | see above | see above | see above | see above |

**Key observations:**
- The rating matrix is >98% sparse — SVD with k=50 factors captures the main preference structure efficiently.
- The long-tail in movie popularity means a small fraction of movies dominate recommendations (popularity bias). The hybrid model reduces this by injecting content diversity.
- Temporal drift in the rating distribution is mild, but should be monitored in production.

**Next steps for production:**
- Periodically retrain CF on fresh ratings (e.g. weekly)
- Log recommendation events and compute online precision/hit-rate
- Introduce a re-ranking layer to enforce business constraints (freshness, diversity, promotional items)